# Testing Inference
Testing para dataset color (RGB).

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º ***(v5i)***
    - Plus soil images
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Experiments:**
    1. Evaluación del modelo para conjunto de testeo.

## Init

In [14]:
import os
import shutil
import fnmatch
import pickle

In [15]:
!pip install ultralytics

## Helper Functions

In [16]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [17]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [18]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [19]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [65]:
import pickle

def save_results(results, filename, verbose = True):
    output_file = f"{filename}.pkl"
    with open(output_file, 'wb') as f:
        pickle.dump(results, f)

    if verbose:
      print(f"✅ Results saved as PKL to {output_file}")

def load_results(filename, verbose = True):
    with open(filename, 'rb') as f:
        return pickle.load(f)

    if verbose:
      print(f"✅ Results loaded as PKL from {filename}")

In [21]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

# Inference functions

In [76]:
import json

def serialize_results(obj):
    if isinstance(obj, YOLO):
        return str(obj)  # Convert YOLO model object to string representation
    elif hasattr(obj, 'tolist'):  # Check if object has tolist() method (e.g., NumPy arrays, tensors)
        return obj.tolist()
    elif isinstance(obj, (str, int, float, bool, type(None))):
        return obj  # Directly serialize basic types
    else:
        try:
            # Attempt to convert object to dictionary if possible
            return obj.__dict__
        except AttributeError:
            raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

def save_as_json(results, filename="results.json", verbose = True):
    output_file = f"{filename}.json"
    serialized_results = []
    for result in results:
        serialized_result = {
            "path": result.path,
            "names": result.names,
            "boxes": serialize_results(result.boxes),
            "masks": serialize_results(result.masks),
            "probs": serialize_results(result.probs),
            "keypoints": serialize_results(result.keypoints),
            "speed": result.speed  # Assuming speed is already JSON serializable
        }
        serialized_results.append(serialized_result)

    with open(output_file, "w") as f:
        json.dump(serialized_results, f, default=serialize_results, indent=4)

    if verbose:
      print(f"✅ Results saved as JSON to {output_file}")


In [75]:
def save_bboxes(results, filename, verbose = True):
    """
    Convierte los resultados de bounding boxes de YOLO a formato COCO TXT.

    Args:
        results (ultralytics.engine.results.Results): Resultados de la inferencia de YOLO.
        output_file (str): Ruta al archivo TXT de salida.
    """
    output_file = f"{filename}.txt"
    with open(output_file, 'w') as f:
        for result in results:
            if result.boxes:
                boxes = result.boxes.xywhn.tolist() # Obtener bounding boxes normalizados (xywhn)
                classes = result.boxes.cls.tolist() # Obtener clases.
                for box, cls in zip(boxes, classes):
                    x_center, y_center, width, height = box
                    f.write(f"{int(cls)} {x_center} {y_center} {width} {height}\n")

    if verbose:
      print(f"✅ Results saved as TXT to {output_file}")

### Validation functions

In [22]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [23]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"❌ An error occurred: {e}")

  #return json_data


In [24]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  print("Total objects detected:", total_det)
  print("Confusion matrix:")
  for row in percentages:
      print(row)

  return matrix


In [25]:
def show_cm(TP, FP, FN):
    matrix = [[TP, FP], [FN, 0]]
    total_det = sum(sum(value) for value in matrix)
    percentages = []
    for row in matrix:
        values_percentages = []
        for value in row:
            if total_det != 0:
                percentage = (value / total_det) * 100
            else:
                percentage = 0.0
            values_percentages.append(f"{percentage:.2f}%")
        percentages.append(values_percentages)

    print("Total objects detected:", total_det)
    print("\nConfusion matrix:")
    for row in percentages:
        a, b = row
        print(f"[ {a} , {b} ]")

In [26]:
def show_metrics(TP, FP, FN):
    show_cm(TP, FP, FN)
    accuracy = TP/(TP+FP+FN)
    precision = TP/(TP+FP)
    recall = TP/(TP+FN)
    f1 = 2 * (precision * recall) / (precision + recall)
    f2 = 1.25 * (precision * recall) / (0.25 * precision + recall)
    fm = (precision * recall) ** 0.5
    print("\nMetrics:")
    print(f"- Accuracy: {accuracy:.3f}")
    print(f"- Precision: {precision:.3f}")
    print(f"- Recall: {recall:.3f}")
    print(f"- F1 Score: {f1:.3f}")
    print(f"- F½ Score: {f2:.3f}")
    print(f"- G-mean: {fm:.3f}")

# Datasets builder

## Importing from Drive

In [28]:
!rm -rf /content/sample_data

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [29]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

 3.5m.v3i.yolov8.640px
 3.5m.v3i.yolov8.640px.aug.v1
 3.5m.v3i.yolov8.640px.aug.v1.soil_aug
 3.5m.v3i.yolov8.640px_clahe
 3.5m.v3i.yolov8.640px.soil_aug
 3.5m.v4i.yolov8.640px
 3.5m.v4i.yolov8.640px_209
 3.5m.v4i.yolov8_blended.640px
 3.5m.v5i.yolov8.640px-2steps.aug2
 3.5m.v5i.yolov8.640px.aug.v1
 3.5m.v5i.yolov8_blended.640px.aug.v1
'7 Validation_experiments_5_(exp_88).ipynb'
 Inference
 models
 optuna_yolov8_f1_study.db
 save_optuna
 TESTING


In [32]:
drive_path = '/content/drive/MyDrive/YOLO/TESTING/'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 5 dataset options:


['Desarrollo', 'lote 212 - anegado', 'Desenfocado', 'Alturas', 'Eryx aro']

In [33]:
choose_dataset = 1 #10
index = choose_dataset - 1
dataset_name = os.listdir(drive_path)[index]
print("Chosen dataset:", dataset_name)

Chosen dataset: Desarrollo


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [35]:
# Option 2 (download just the needed dataset)
cloud_path = f"{drive_path}/{dataset_name}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path

In [36]:
src_folder = f"/content/YOLO/{dataset_name}"

In [ ]:
# Option 3 (download just what's needed)
!mkdir '/content/YOLO/'
for split in ['valid', 'test', 'small', 'large']:
  cloud_path = f"{drive_path}/{dataset_name}/{split}/"
  local_path = f"/content/YOLO/{dataset_name}/"
  !mkdir $local_path
  !cp -r $cloud_path $local_path
!cp -r $yaml_path $local_path

---

In [39]:
drive_path = '/content/drive/MyDrive/YOLO/'
models_path = f'{drive_path}/models'
drive_models_path = os.listdir(models_path)
drive_models = len(drive_models_path)
if (drive_models) > 1:
    print("There are %d dataset options:" % drive_models)
else:
    print("Theres is only 1 dataset:")
drive_models_path

There are 5 dataset options:


['best_e26.pt', 'best_e68.pt', 'best_e50.pt', 'best_e86.pt', 'best_e79.pt']

In [40]:
choose_model = 4
index = choose_model - 1
model_name = os.listdir(models_path)[index]
print("Chosen model:", model_name)

Chosen model: best_e86.pt


In [41]:
# Option 2 (download just the dataset needed)
model_cloud_path = f"{models_path}/{model_name}"
model_local_path = f"/content/YOLO/"
!cp -r $model_cloud_path $model_local_path
model_weights = f"/content/YOLO/{model_name}"

In [42]:
import re
match = re.search(r"e(\d+)\.", model_name)

if match:
    model_num = match.group(1)
    model_num = f"e{model_num}"
    print(model_num)
else:
    print("No se encontró el número en el nombre del archivo.")

e86


## Setup

In [43]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [44]:
models = []

In [45]:
# Load currently trained YOLO model
model = YOLO(model_weights)
models.append(model)

In [46]:
# Best values found with Optuna
iou = 0.326691441
conf = 0.261453146

### Optimization

In [47]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [48]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [49]:
!nvidia-smi

Sat May 17 00:45:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [50]:
!yolo version

8.3.137


# Experimentation

-----
## Inferences
### *YOLOv8 Mid | Testing*
Color images dataset

In [81]:
project = "testing"

In [83]:
data_options_path = os.listdir(src_folder)
data_options = len(data_options_path)
if (data_options) > 1:
    print("There are %d dataset options:" % data_options)
else:
    print("Theres is only 1 dataset:")
data_options_path

There are 3 dataset options:


['209_415_13', '.DS_Store', '503_119_43']

In [84]:
choose_data = 1 #10
index = choose_data - 1
data_name = os.listdir(src_folder)[index]
print("Chosen dataset:", data_name)
data = f"/content/YOLO/{dataset_name}/{data_name}"

Chosen dataset: 209_415_13


In [94]:
experiment=f"des_{data_name}_"
print("Experimento: ", experiment)
print("Source:", src_folder)

Experimento:  des_209_415_13_
Source: /content/YOLO/Desarrollo


# Inference
Se realizarán las inferencias para cada imagen de forma iterativa (con cada modelo)

In [95]:
files_paths = os.listdir(data)
files_paths.sort()
total_files = len(files_paths)
print(f'Se detectaron un total de {total_files} imágenes:\n')
files_paths

Se detectaron un total de 54 imágenes:



['209_415_13.tile00x00.jpg',
 '209_415_13.tile00x01.jpg',
 '209_415_13.tile00x02.jpg',
 '209_415_13.tile00x03.jpg',
 '209_415_13.tile00x04.jpg',
 '209_415_13.tile00x05.jpg',
 '209_415_13.tile00x06.jpg',
 '209_415_13.tile00x07.jpg',
 '209_415_13.tile00x08.jpg',
 '209_415_13.tile01x00.jpg',
 '209_415_13.tile01x01.jpg',
 '209_415_13.tile01x02.jpg',
 '209_415_13.tile01x03.jpg',
 '209_415_13.tile01x04.jpg',
 '209_415_13.tile01x05.jpg',
 '209_415_13.tile01x06.jpg',
 '209_415_13.tile01x07.jpg',
 '209_415_13.tile01x08.jpg',
 '209_415_13.tile02x00.jpg',
 '209_415_13.tile02x01.jpg',
 '209_415_13.tile02x02.jpg',
 '209_415_13.tile02x03.jpg',
 '209_415_13.tile02x04.jpg',
 '209_415_13.tile02x05.jpg',
 '209_415_13.tile02x06.jpg',
 '209_415_13.tile02x07.jpg',
 '209_415_13.tile02x08.jpg',
 '209_415_13.tile03x00.jpg',
 '209_415_13.tile03x01.jpg',
 '209_415_13.tile03x02.jpg',
 '209_415_13.tile03x03.jpg',
 '209_415_13.tile03x04.jpg',
 '209_415_13.tile03x05.jpg',
 '209_415_13.tile03x06.jpg',
 '209_415_13.t

In [96]:
# Garbage collection
import gc
for i in range(20):
  torch.cuda.empty_cache()
  gc.collect()

In [97]:
import time

def start_stopwatch():
  start_time = time.time()  # Momento de inicio
  return start_time

def end_stopwatch(start_time):
  end_time = time.time()  # Momento de fin
  return end_time - start_time  # Calcula el tiempo de ejecución

In [98]:
import os
import cv2

def count_plants(model, file_name, verbose=False):
  count: int = 0

  # Realiza inferencia para la imagen
  image_path = os.path.join(src_folder, data, file_name)  # Use os.path.join for platform compatibility

  # Validate image before processing
  if not os.path.exists(image_path):
    print(f"WARNING ⚠️ Image file not found: {image_path}")
    return 0  # Skip if file not found

  try:
    image = cv2.imread(image_path)  # Read image using cv2.imread directly
    if image is None:
      print(f"WARNING ⚠️ Image Read Error {image_path}")
      return 0  # Skip if image could not be read
    results = model(image, save=True, project=project, name=experiment, conf=conf, iou=iou, verbose=verbose) # INFERENCIA
  except Exception as e:
    print(f"Error processing image {image_path}: {e}")
    return 0  # Skip if any error occurs

  # Acceder a las coordenadas de los bounding boxes
  for result in results:
      print(result.verbose()) if verbose else ''
      for box in result.boxes:
          count +=1

  # Se almacena cada inferencia como JSON y PKL (log)
  file = os.path.splitext(file_name)[0]
  save_as = f"/content/{project}/{experiment}/inference.{file}"
  save_results(results, save_as, verbose = verbose)
  save_as_json(results, save_as, verbose = verbose)
  save_bboxes(results, save_as, verbose = verbose)

  return count

In [99]:
import pandas as pd

# Ejecución en batch para todos los mosaicos
verbose = False # Activar para incluir más información sobre almacenamiento de predicciones
total_time = []
total_count = []
model = None
plant_count = 0
for count, file_name in enumerate(files_paths):

    print(f"❇️ Procesado {round((count+1)/total_files*100,3):.2f}%\n") if count > 0 else ''
    print(f"INFERENCE {count+1}/{total_files}:\n {file_name}")
    print()
    infer_time = []
    infer_count = []

    for index, model in enumerate(models):

        # Setup del entrenamiento
        project= f"model{index}"
        print(f"Proyecto: {project}")
        timer = start_stopwatch() # Inicia medición de tiempo
        plant_count = count_plants(model, file_name) # Ejecuta la inferencia y guarda los datos
        measured_time = end_stopwatch(timer) # Termina medición de tiempo
        print(f"  Tiempo de inferencia {(measured_time*1000):.4} ms")
        print(f"  Se detectaron {plant_count} objetos plant-weed.")
        infer_time.append(measured_time) # Acumula el histórico de rendimiento
        infer_count.append(plant_count) # Acumula el histórico del conteo
        print()

    total_time.append(infer_time)
    total_count.append(infer_count)
    print() if verbose else ''

df_count = pd.DataFrame(total_count)
df_time = pd.DataFrame(total_time)
print(f"✅ EJECUCIÓN COMPLETA\n")

count_sums = df_count.sum().values
time_sums = df_time.sum().values

print("Totales por modelo:")
for index, (time_, count_) in enumerate(zip(time_sums, count_sums)):
    print(f"  Modelo {index + 1} ({float(time_):.2f} seg): {int(count_)}")
print()

INFERENCE 1/54:
 209_415_13.tile00x00.jpg

Proyecto: model0
Results saved to model0/des_209_415_13_
  Tiempo de inferencia 58.31 ms
  Se detectaron 13 objetos plant-weed.

❇️ Procesado 3.70%

INFERENCE 2/54:
 209_415_13.tile00x01.jpg

Proyecto: model0
Results saved to model0/des_209_415_13_2
  Tiempo de inferencia 57.94 ms
  Se detectaron 28 objetos plant-weed.

❇️ Procesado 5.56%

INFERENCE 3/54:
 209_415_13.tile00x02.jpg

Proyecto: model0
Results saved to model0/des_209_415_13_3
  Tiempo de inferencia 55.33 ms
  Se detectaron 16 objetos plant-weed.

❇️ Procesado 7.41%

INFERENCE 4/54:
 209_415_13.tile00x03.jpg

Proyecto: model0
Results saved to model0/des_209_415_13_4
  Tiempo de inferencia 59.32 ms
  Se detectaron 13 objetos plant-weed.

❇️ Procesado 9.26%

INFERENCE 5/54:
 209_415_13.tile00x04.jpg

Proyecto: model0
Results saved to model0/des_209_415_13_5
  Tiempo de inferencia 58.36 ms
  Se detectaron 17 objetos plant-weed.

❇️ Procesado 11.11%

INFERENCE 6/54:
 209_415_13.tile00x

In [100]:
count_avg = df_count.mean().values
time_avg = df_time.mean().values

print("Promedios por modelo:")
for index, (time_, count_) in enumerate(zip(time_avg, count_avg)):
    print(f"  Modelo {index + 1} ({float(time_*1000):.3f} ms): {float(count_):.1f} obj/image")
print()

Promedios por modelo:
  Modelo 1 (40.073 ms): 20.5 obj/image



# Almacenamiento de predicciones

In [103]:
# Store experiments
!mkdir /content/drive/MyDrive/Results
for index, model in enumerate(models):
    project= f"model{index}"
    save_on_cloud(source=f'/content/{project}/{experiment}/', destination=f'/content/drive/MyDrive/Results/{project}/{experiment}/')
    print()



mkdir: cannot create directory ‘/content/drive/MyDrive/Results’: File exists
✅ Folder copied successfully:
   /content/model0/des_209_415_13_/ 
  --> /content/drive/MyDrive/Results/model0/des_209_415_13_/



In [104]:
# Save execution time log
filenames = [{
    'file': df_time,
    'name': f"{experiment}.total_time.csv"},
     {'file': df_count,
    'name': f"{experiment}.total_count.csv"}
  ]

for filename in filenames:
  try:
      name = filename['name']
      df = filename['file']
      if df is not None:
        df.to_csv(f"/content/{name}")
      print(f"✅ Execution log saved as CSV to {name}")

      source=f'/content/{name}'
      destination='/content/drive/MyDrive/Results/'
      !cp $source $destination

  except PermissionError:
      print(f"Error: Permission denied. Could not save {name}")
  except OSError as e:
      print(f"Error: An operating system error occurred: {e} when trying to save {name}.")
  except Exception as e:
      print(f"An unexpected error occurred: {e} when trying to save {name}.")

✅ Execution log saved as CSV to des_209_415_13_.total_time.csv
✅ Execution log saved as CSV to des_209_415_13_.total_count.csv
